# V18 BCR/TCR Donor-Level IT-Oriented Analysis
**Date:** 2026-03-14
**Component:** C12 — BCR/TCR Repertoire Analysis
**Method:** Donor-level Mann-Whitney U (consistent with C3-C10 pipeline)
**Key questions:**
1. Is plasmaB_c01-SDC1 collapse donor-consistent at NL→IT?
2. B cell subcluster redistribution — which are IT-specific?
3. BCR clonality, isotype shift, V-gene usage — tissue-separated donor-level
4. TCR clonality — tissue-separated donor-level
5. Pattern classification: IT-specific vs chronic-persistent

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

def safe_clonality(clone_counts):
    nu = len(clone_counts)
    nt = clone_counts.sum()
    if nu <= 0 or nt <= 0: return 0.0
    if nu == 1: return 1.0 if nt > 1 else 0.0
    fr = clone_counts.values / nt
    fr = fr[fr > 0]
    ent = -np.sum(fr * np.log2(fr))
    return 1 - (ent / np.log2(nu)) if np.log2(nu) > 0 else 0.0

def mw_test(nl_vals, it_vals, label=''):
    """Mann-Whitney with consistency count."""
    nl_v = nl_vals.dropna()
    it_v = it_vals.dropna()
    if len(nl_v) < 2 or len(it_v) < 2:
        return None
    stat, p = mannwhitneyu(nl_v, it_v, alternative='two-sided')
    nl_m, it_m = nl_v.mean(), it_v.mean()
    d = '↑' if it_m > nl_m else '↓'
    pct = ((it_m - nl_m) / nl_m * 100) if nl_m != 0 else float('inf')
    # Donor-pair consistency
    pairs_total = 0; pairs_consistent = 0
    for nv in nl_v:
        for iv in it_v:
            pairs_total += 1
            if (it_m > nl_m and iv > nv) or (it_m <= nl_m and iv <= nv):
                pairs_consistent += 1
    consistency = f'{pairs_consistent}/{pairs_total}'
    sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
    return {'metric': label, 'NL_mean': nl_m, 'IT_mean': it_m,
            'direction': d, 'pct_change': pct, 'p_value': p,
            'consistency': consistency, 'sig': sig}

print(f'Total cells: {len(obs):,}')
print(f'Donors: {obs["donor"].nunique()}')
print('Setup complete.')

In [ ]:
# Cell 2: B/PlasmaB SUBCLUSTER PROPORTIONS — Donor-Level
# Key: proportion of each subcluster within total B+PlasmaB cells per donor per tissue
print('='*70)
print('SECTION A: B/PlasmaB Subcluster Proportions (Donor-Level)')
print('='*70)

b_plasma = obs[obs['major_lineage'].isin(['B', 'PlasmaB'])].copy()
subclusters = sorted(b_plasma['gut2021_subcluster_v2'].unique())
print(f'B/PlasmaB subclusters: {subclusters}')
print(f'Total B/PlasmaB cells: {len(b_plasma):,}')

# Calculate donor-level proportions
prop_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    total = len(grp)
    if total < 5:  # skip donors with very few B cells
        continue
    sc_counts = grp['gut2021_subcluster_v2'].value_counts()
    for sc in subclusters:
        count = sc_counts.get(sc, 0)
        prop_rows.append({
            'Stage': stage, 'tissue': tissue, 'donor': donor,
            'subcluster': sc, 'count': count, 'total_bp': total,
            'proportion': count / total * 100
        })

prop_df = pd.DataFrame(prop_rows)
print(f'Donor-level proportion records: {len(prop_df)}')

# Mann-Whitney NL→IT and NL→IA for each subcluster × tissue
results_a = []
for tissue_val in ['Liver', 'Blood']:
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()}: B/PlasmaB Subcluster Proportions NL→IT')
    print(f'{"─"*70}')
    for sc in subclusters:
        sub = prop_df[(prop_df['tissue'] == tissue_val) & (prop_df['subcluster'] == sc)]
        nl = sub[sub['Stage'] == 'NL']['proportion']
        it = sub[sub['Stage'] == 'IT']['proportion']
        ia = sub[sub['Stage'] == 'IA']['proportion']
        
        # NL→IT
        r_it = mw_test(nl, it, f'{sc}')
        # NL→IA
        r_ia = mw_test(nl, ia, f'{sc}')
        
        if r_it:
            # Pattern classification
            it_sig = r_it['p_value'] < 0.05
            ia_sig = r_ia['p_value'] < 0.05 if r_ia else False
            if it_sig and not ia_sig:
                pattern = 'IT-specific'
            elif it_sig and ia_sig:
                pattern = 'Chronic-persistent'
            elif not it_sig and ia_sig:
                pattern = 'IA-emergent'
            else:
                pattern = 'NS'
            
            r_it['tissue'] = tissue_val
            r_it['NL_IA_p'] = r_ia['p_value'] if r_ia else np.nan
            r_it['pattern'] = pattern
            results_a.append(r_it)
            
            ia_p_str = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
            print(f'  {r_it["sig"]} {sc}: NL={r_it["NL_mean"]:.1f}% → IT={r_it["IT_mean"]:.1f}% '
                  f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                  f'[{r_it["consistency"]}] | NL→IA {ia_p_str} | {pattern}')

results_a_df = pd.DataFrame(results_a)
results_a_df.to_csv(f'{SAVE_DIR}/A_subcluster_proportions_MW.csv', index=False)
print(f'\nSaved: A_subcluster_proportions_MW.csv')

In [ ]:
# Cell 3: PlasmaB/B RATIO — Donor-Level (differentiation efficiency)
print('='*70)
print('SECTION B: PlasmaB/B Ratio = Differentiation Efficiency')
print('='*70)

ratio_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    n_b = (grp['major_lineage'] == 'B').sum()
    n_pb = (grp['major_lineage'] == 'PlasmaB').sum()
    n_sdc1 = (grp['gut2021_subcluster_v2'] == 'plasmaB_c01-SDC1').sum()
    total = n_b + n_pb
    if total < 5:
        continue
    ratio_rows.append({
        'Stage': stage, 'tissue': tissue, 'donor': donor,
        'n_B': n_b, 'n_PlasmaB': n_pb, 'n_SDC1': n_sdc1,
        'total_BP': total,
        'PlasmaB_ratio': n_pb / total * 100,
        'SDC1_ratio': n_sdc1 / total * 100,
        'SDC1_of_PlasmaB': n_sdc1 / n_pb * 100 if n_pb > 0 else 0,
    })

ratio_df = pd.DataFrame(ratio_rows)

results_b = []
for tissue_val in ['Liver', 'Blood']:
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()}')
    print(f'{"─"*70}')
    t = ratio_df[ratio_df['tissue'] == tissue_val]
    for stage in ['NL', 'IT', 'IA', 'AR', 'CR']:
        s = t[t['Stage'] == stage]
        if len(s) == 0:
            continue
        print(f'  {stage} ({len(s)} donors): PlasmaB ratio={s.PlasmaB_ratio.mean():.1f}%, '
              f'SDC1 ratio={s.SDC1_ratio.mean():.1f}%, '
              f'SDC1/PlasmaB={s.SDC1_of_PlasmaB.mean():.1f}%')
    
    # Mann-Whitney
    for metric in ['PlasmaB_ratio', 'SDC1_ratio', 'SDC1_of_PlasmaB']:
        nl = t[t['Stage'] == 'NL'][metric]
        it = t[t['Stage'] == 'IT'][metric]
        ia = t[t['Stage'] == 'IA'][metric]
        r_it = mw_test(nl, it, f'{metric}')
        r_ia = mw_test(nl, ia, f'{metric}')
        if r_it:
            it_sig = r_it['p_value'] < 0.05
            ia_sig = r_ia['p_value'] < 0.05 if r_ia else False
            pattern = 'IT-specific' if it_sig and not ia_sig else \
                      'Chronic' if it_sig and ia_sig else \
                      'IA-emergent' if not it_sig and ia_sig else 'NS'
            r_it['tissue'] = tissue_val; r_it['pattern'] = pattern
            results_b.append(r_it)
            ia_p = r_ia['p_value'] if r_ia else np.nan
            print(f'  {r_it["sig"]} {metric}: NL={r_it["NL_mean"]:.2f}→IT={r_it["IT_mean"]:.2f} '
                  f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                  f'[{r_it["consistency"]}] NL→IA p={ia_p:.4f} | {pattern}')

pd.DataFrame(results_b).to_csv(f'{SAVE_DIR}/B_differentiation_ratio_MW.csv', index=False)
ratio_df.to_csv(f'{SAVE_DIR}/B_donor_level_ratios.csv', index=False)
print(f'\nSaved: B_differentiation_ratio_MW.csv, B_donor_level_ratios.csv')

In [ ]:
# Cell 4: BCR REPERTOIRE — Donor-Level (clonality, isotype, V-gene)
print('='*70)
print('SECTION C: BCR Repertoire Donor-Level Metrics')
print('='*70)

def donor_bcr_full(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_bcr':0,'pct_bcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,'pct_IgD':np.nan,
                         'pct_switched':np.nan,'top_clone':0,
                         'pct_IGHV3_23':np.nan,'n_unique_vgenes':0})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        clon = safe_clonality(cc)
        nu = len(cc); ns = (cc==1).sum()
        iso = bcr['BCR_CType'].value_counts(); it = iso.sum()
        igm = iso.get('IGHM',0)/it*100; igg = iso.get('IGHG',0)/it*100
        iga = iso.get('IGHA',0)/it*100; igd = iso.get('IGHD',0)/it*100
        vg = bcr['BCR_v_gene'].value_counts()
        pct_v3_23 = vg.get('IGHV3-23',0)/nb*100
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_bcr':nb,'pct_bcr':nb/n*100,
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,'pct_IgD':igd,
                     'pct_switched':igg+iga,'top_clone':cc.max(),
                     'pct_IGHV3_23':pct_v3_23,'n_unique_vgenes':len(vg)})
    return pd.DataFrame(rows)

bcr_results = []
for tissue_val in ['Liver','Blood']:
    df = donor_bcr_full(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — BCR Repertoire')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_bcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no BCR'); continue
        print(f'  {stage} ({len(s)}d): BCR={s.n_bcr.mean():.0f}, '
              f'clon={s.clonality.mean():.4f}, sing={s.pct_singleton.mean():.1f}%, '
              f'IgM={s.pct_IgM.mean():.1f}% IgG={s.pct_IgG.mean():.1f}% '
              f'IgA={s.pct_IgA.mean():.1f}% IgD={s.pct_IgD.mean():.1f}% '
              f'sw={s.pct_switched.mean():.1f}% V3-23={s.pct_IGHV3_23.mean():.1f}%')
    
    # Mann-Whitney NL→IT and NL→IA
    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_bcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_IgD',
                  'pct_switched','pct_bcr','top_clone','pct_IGHV3_23']:
            r_it = mw_test(nl[m], it[m], m)
            r_ia = mw_test(nl[m], ia[m], m) if len(ia)>=2 else None
            if r_it:
                it_sig = r_it['p_value']<0.05
                ia_sig = r_ia['p_value']<0.05 if r_ia else False
                pattern = 'IT-spec' if it_sig and not ia_sig else \
                          'Chronic' if it_sig and ia_sig else \
                          'IA-emrg' if not it_sig and ia_sig else 'NS'
                ia_p = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
                r_it['tissue']=tissue_val; r_it['pattern']=pattern
                bcr_results.append(r_it)
                print(f'    {r_it["sig"]} {m}: {r_it["NL_mean"]:.2f}→{r_it["IT_mean"]:.2f} '
                      f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                      f'[{r_it["consistency"]}] | IA:{ia_p} | {pattern}')

pd.DataFrame(bcr_results).to_csv(f'{SAVE_DIR}/C_BCR_repertoire_MW.csv', index=False)
# Save donor-level data
bcr_full = pd.concat([donor_bcr_full(obs,'Liver'), donor_bcr_full(obs,'Blood')], ignore_index=True)
bcr_full.to_csv(f'{SAVE_DIR}/C_BCR_donor_level_full.csv', index=False)
print(f'\nSaved: C_BCR_repertoire_MW.csv, C_BCR_donor_level_full.csv')

In [ ]:
# Cell 5: TCR REPERTOIRE — Donor-Level
print('='*70)
print('SECTION D: TCR Repertoire Donor-Level Metrics')
print('='*70)

def donor_tcr_full(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_tcr':0,'pct_tcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'top_clone':0,'n_unique_clones':0,
                         'top_vgene':'','top_vgene_pct':np.nan})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        clon = safe_clonality(cc)
        nu = len(cc); ns = (cc==1).sum()
        vg = tcr['TCR_v_gene.x'].value_counts()
        top_vg = vg.index[0] if len(vg)>0 else ''
        top_vg_pct = vg.iloc[0]/nt*100 if len(vg)>0 else 0
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_tcr':nt,'pct_tcr':nt/n*100,
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'top_clone':cc.max(),'n_unique_clones':nu,
                     'top_vgene':top_vg,'top_vgene_pct':top_vg_pct})
    return pd.DataFrame(rows)

tcr_results = []
for tissue_val in ['Liver','Blood']:
    df = donor_tcr_full(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — TCR Repertoire')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_tcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no TCR'); continue
        print(f'  {stage} ({len(s)}d): TCR={s.n_tcr.mean():.0f}, '
              f'clon={s.clonality.mean():.4f}, sing={s.pct_singleton.mean():.1f}%, '
              f'top_clone={s.top_clone.mean():.0f}, unique={s.n_unique_clones.mean():.0f}')
    
    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_tcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_tcr','top_clone','n_unique_clones']:
            r_it = mw_test(nl[m], it[m], m)
            r_ia = mw_test(nl[m], ia[m], m) if len(ia)>=2 else None
            if r_it:
                it_sig = r_it['p_value']<0.05
                ia_sig = r_ia['p_value']<0.05 if r_ia else False
                pattern = 'IT-spec' if it_sig and not ia_sig else \
                          'Chronic' if it_sig and ia_sig else \
                          'IA-emrg' if not it_sig and ia_sig else 'NS'
                ia_p = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
                r_it['tissue']=tissue_val; r_it['pattern']=pattern
                tcr_results.append(r_it)
                print(f'    {r_it["sig"]} {m}: {r_it["NL_mean"]:.3f}→{r_it["IT_mean"]:.3f} '
                      f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                      f'[{r_it["consistency"]}] | IA:{ia_p} | {pattern}')

pd.DataFrame(tcr_results).to_csv(f'{SAVE_DIR}/D_TCR_repertoire_MW.csv', index=False)
tcr_full = pd.concat([donor_tcr_full(obs,'Liver'), donor_tcr_full(obs,'Blood')], ignore_index=True)
tcr_full.to_csv(f'{SAVE_DIR}/D_TCR_donor_level_full.csv', index=False)
print(f'\nSaved: D_TCR_repertoire_MW.csv, D_TCR_donor_level_full.csv')

In [ ]:
# Cell 6: COMPREHENSIVE SUMMARY TABLE
print('='*70)
print('COMPREHENSIVE SUMMARY: ALL SIGNIFICANT + TREND FINDINGS')
print('='*70)

all_results = []
for r in results_a:  # subcluster proportions
    r['section'] = 'A_subcluster_prop'
    all_results.append(r)
for r in results_b:  # differentiation ratios
    r['section'] = 'B_diff_ratio'
    all_results.append(r)
for r in bcr_results:  # BCR repertoire
    r['section'] = 'C_BCR_repertoire'
    all_results.append(r)
for r in tcr_results:  # TCR repertoire
    r['section'] = 'D_TCR_repertoire'
    all_results.append(r)

all_df = pd.DataFrame(all_results)

# Show significant and trend findings
sig_df = all_df[all_df['p_value'] < 0.10].sort_values('p_value')
print(f'\nFindings with p < 0.10: {len(sig_df)}')
print(f'Findings with p < 0.05: {len(all_df[all_df["p_value"]<0.05])}')
print()
for _, r in sig_df.iterrows():
    print(f'{r["sig"]} [{r["section"]}] {r["tissue"]}/{r["metric"]}: '
          f'NL={r["NL_mean"]:.2f}→IT={r["IT_mean"]:.2f} '
          f'({r["direction"]}{abs(r["pct_change"]):.1f}%) '
          f'p={r["p_value"]:.4f} [{r["consistency"]}] | {r["pattern"]}')

all_df.to_csv(f'{SAVE_DIR}/MASTER_all_MW_results.csv', index=False)
sig_df.to_csv(f'{SAVE_DIR}/MASTER_significant_results.csv', index=False)
print(f'\nSaved: MASTER_all_MW_results.csv ({len(all_df)} tests)')
print(f'Saved: MASTER_significant_results.csv ({len(sig_df)} findings)')

# IT-specific findings
it_spec = all_df[all_df['pattern'].str.contains('IT-spec', na=False)]
print(f'\n--- IT-Specific findings: {len(it_spec)} ---')
for _, r in it_spec.iterrows():
    print(f'  {r["sig"]} {r["tissue"]}/{r["metric"]}: p={r["p_value"]:.4f} [{r["consistency"]}]')

print(f'\n✅ All C12 analysis complete. Results in: {SAVE_DIR}')